# Brain Tumor Detection Using Custom CNN
### Internship Assignment Submission

This notebook trains a custom Convolutional Neural Network (CNN) from scratch to classify cranial MRI scans into four classes: Glioma, Meningioma, Pituitary Tumor, and No Tumor.

## 1. Import Dependencies

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

print("TensorFlow Version:", tf.__version__)

## 2. Load Dataset

In [ ]:
DATASET_PATH = "dataset"
IMAGE_SIZE = 128
CLASSES = ["glioma", "meningioma", "no_tumor", "pituitary"]

images = []
labels = []

for class_index, class_name in enumerate(CLASSES):
    class_path = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_path):
        continue
    print(f"Loading {class_name} images...")
    for image_name in os.listdir(class_path):
        image_path = os.path.join(class_path, image_name)
        image = cv2.imread(image_path)
        if image is None:
            continue
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image / 255.0
        images.append(image)
        labels.append(class_index)

images = np.array(images, dtype="float32")
labels = np.array(labels)
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

## 3. Split Dataset (70% Train, 15% Val, 15% Test)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.30, random_state=42, stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

y_train_cat = to_categorical(y_train, num_classes=4)
y_val_cat = to_categorical(y_val, num_classes=4)
y_test_cat = to_categorical(y_test, num_classes=4)

print("Training set:", len(X_train))
print("Validation set:", len(X_val))
print("Testing set:", len(X_test))

## 4. Define Data Augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
    tf.keras.layers.RandomTranslation(height_factor=0.10, width_factor=0.10),
    tf.keras.layers.RandomBrightness(factor=0.10)
])

## 5. Build Custom CNN Model

In [ ]:
model = Sequential([
    tf.keras.layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    data_augmentation,
    
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.30),
    
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.30),
    
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.50),
    Dense(256, activation='relu'),
    Dropout(0.40),
    Dense(128, activation='relu'),
    Dropout(0.30),
    Dense(4, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 6. Model Training

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# Set epochs to a small number if just running to verify, or 30-50 for full training
EPOCHS = 30
BATCH_SIZE = 32

print("Ready to train model. Run model.fit() in this cell.")